In [35]:
import json
import time
from pathlib import Path

import cv2
import yaml
from ultralytics import YOLO


In [36]:
# =========================================================
# RESUMABLE YOLO26 TRAINING (YOLO26M-READY)
# =========================================================

# -----------------------------
# PATHS
# -----------------------------
PROJECT_ROOT = Path(r"C:\Users\benne\Computer_Vision\BoxingPunchDetection")
RAW_DATA_ROOT = PROJECT_ROOT / "raw_data"
DATASET_ROOT = PROJECT_ROOT / "datasets" / "boxing_yolo26_full"
MODELS_ROOT = PROJECT_ROOT / "yolo26_models"

# -----------------------------
# SETTINGS
# -----------------------------
MODEL_NAME = "yolo26l.pt"
EPOCHS = 20
IMGSZ = 640
BATCH = 2                   
WORKERS = 0
DEVICE = 0                  
RUN_PREDICTION_AFTER_TRAIN = False

FORCE_NEW_RUN = False


ALLOWED_MODELS = {
    "yolo26n.pt", "yolo26s.pt", "yolo26m.pt", "yolo26l.pt", "yolo26x.pt",
    "yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov8l.pt", "yolov8x.pt",
}
MODEL_STEM = Path(MODEL_NAME).stem
RUN_NAME = f"resume_{MODEL_STEM}"


In [37]:

POLISH_TO_CLASS = {
    "Głowa lewą ręką": 0,
    "Głowa prawą ręką": 1,
    "Korpus lewą ręką": 2,
    "Korpus prawą ręką": 3,
    "Blok lewą ręką": 4,
    "Blok prawą ręką": 5,
    "Chybienie lewą ręką": 6,
    "Chybienie prawą ręką": 7,
}

CLASS_NAMES = [
    "left_head",
    "right_head",
    "left_body",
    "right_body",
    "left_block",
    "right_block",
    "left_miss",
    "right_miss",
]


In [38]:

# =========================================================
# UTILS
# =========================================================
def fmt(t):
    return f"{t:.2f}s ({t/60:.2f} min)"


def ensure_exists(p, name):
    if not p.exists():
        raise FileNotFoundError(f"{name} not found: {p}")


def validate_settings():
    if MODEL_NAME not in ALLOWED_MODELS:
        raise ValueError(
            f"Unsupported MODEL_NAME: {MODEL_NAME}\n"
            f"Choose one of: {sorted(ALLOWED_MODELS)}"
        )



In [39]:

# =========================================================
# DATASET CHECK
# =========================================================
def dataset_ready(root):
    img_dir = root / "images" / "train"
    lbl_dir = root / "labels" / "train"
    yaml_file = root / "data.yaml"

    if not img_dir.exists() or not lbl_dir.exists() or not yaml_file.exists():
        return False

    imgs = list(img_dir.glob("*.jpg"))
    lbls = list(lbl_dir.glob("*.txt"))

    if len(imgs) == 0 or len(lbls) == 0:
        return False

    if len(imgs) != len(lbls):
        print("Mismatch between images and labels -> rebuilding", flush=True)
        return False

    print(f"Dataset ready: {len(imgs)} images", flush=True)
    return True



In [40]:

# =========================================================
# DATASET BUILD
# =========================================================
def find_pairs(root):
    pairs = []
    for d in root.iterdir():
        if not d.is_dir():
            continue

        ann = d / "annotations.json"
        vid_dir = d / "data"

        if not ann.exists() or not vid_dir.exists():
            continue

        for v in vid_dir.glob("*.mp4"):
            pairs.append((v, ann, d.name))

    return pairs


def load_tracks(p):
    with open(p, "r", encoding="utf-8") as f:
        d = json.load(f)
    if isinstance(d, list):
        d = d[0]
    return d["tracks"]


def yolo_box(x1, y1, x2, y2, w, h):
    if x2 <= x1 or y2 <= y1:
        return None
    return (
        ((x1 + x2) / 2) / w,
        ((y1 + y2) / 2) / h,
        (x2 - x1) / w,
        (y2 - y1) / h,
    )


def parse_ann(p, w, h):
    tracks = load_tracks(p)
    out = {}

    for t in tracks:
        label = t.get("label")
        if label not in POLISH_TO_CLASS:
            continue

        cls = POLISH_TO_CLASS[label]

        for s in t.get("shapes", []):
            if s.get("outside") or s.get("type") != "rectangle":
                continue

            pts = s.get("points")
            frame_idx = s.get("frame")

            if pts is None or frame_idx is None or len(pts) != 4:
                continue

            box = yolo_box(*pts, w, h)
            if not box:
                continue

            xc, yc, bw, bh = box
            out.setdefault(frame_idx, []).append(
                f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
            )

    return out


def extract_dataset(pairs, root):
    print("Building dataset...", flush=True)
    t0 = time.time()

    img_dir = root / "images" / "train"
    lbl_dir = root / "labels" / "train"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    new = 0

    for vid, ann, name in pairs:
        print(f"Processing video: {vid.name}", flush=True)

        cap = cv2.VideoCapture(str(vid))
        if not cap.isOpened():
            print(f"Could not open video: {vid}", flush=True)
            continue

        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        labels = parse_ann(ann, w, h)

        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if i in labels:
                stem = f"{name}_{i:06d}"
                img = img_dir / f"{stem}.jpg"
                lbl = lbl_dir / f"{stem}.txt"

                if not img.exists():
                    cv2.imwrite(str(img), frame)
                    lbl.write_text("\n".join(labels[i]), encoding="utf-8")
                    new += 1

            i += 1

        cap.release()

    print(f"Dataset built. New images: {new} | time: {fmt(time.time() - t0)}", flush=True)


def write_yaml(root):
    p = root / "data.yaml"
    with open(p, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            {
                "path": str(root),
                "train": "images/train",
                "val": "images/train",
                "names": CLASS_NAMES,
            },
            f,
            sort_keys=False,
            allow_unicode=True,
        )
    return p



In [41]:

# =========================================================
# TRAIN
# =========================================================
def train(data_yaml):
    print("\nStarting training function...", flush=True)

    run_dir = MODELS_ROOT / RUN_NAME
    weights_dir = run_dir / "weights"
    last_ckpt = weights_dir / "last.pt"
    best_ckpt = weights_dir / "best.pt"

    print(f"Model selected: {MODEL_NAME}", flush=True)
    print(f"Run directory: {run_dir}", flush=True)
    print(f"Checking checkpoint: {last_ckpt}", flush=True)

    if FORCE_NEW_RUN and run_dir.exists():
        import shutil
        print("Deleting old run because FORCE_NEW_RUN=True", flush=True)
        shutil.rmtree(run_dir)

    if last_ckpt.exists():
        print("\nRESUMING TRAINING", flush=True)
        print(f"Resuming from: {last_ckpt}", flush=True)

        model = YOLO(str(last_ckpt))
        model.train(resume=True, workers=0, device=DEVICE)

    else:
        print("\nSTARTING NEW TRAINING", flush=True)
        print(f"Starting from base weights: {MODEL_NAME}", flush=True)

        model = YOLO(MODEL_NAME)
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            project=str(MODELS_ROOT),
            name=RUN_NAME,
            val=False,
            workers=WORKERS,
            device=DEVICE,
            verbose=True,
            exist_ok=True,
        )

    print("Training call finished", flush=True)

    if best_ckpt.exists():
        return best_ckpt
    if last_ckpt.exists():
        return last_ckpt

    raise FileNotFoundError(f"No trained weights found in {weights_dir}")



In [42]:

# =========================================================
# TRAIN
# =========================================================
def train(data_yaml):
    print("\nStarting training function...", flush=True)

    run_dir = MODELS_ROOT / RUN_NAME
    weights_dir = run_dir / "weights"
    last_ckpt = weights_dir / "last.pt"
    best_ckpt = weights_dir / "best.pt"

    print(f"Model selected: {MODEL_NAME}", flush=True)
    print(f"Run directory: {run_dir}", flush=True)
    print(f"Checking checkpoint: {last_ckpt}", flush=True)

    if FORCE_NEW_RUN and run_dir.exists():
        import shutil
        print("Deleting old run because FORCE_NEW_RUN=True", flush=True)
        shutil.rmtree(run_dir)

    if last_ckpt.exists():
        print("\nRESUMING TRAINING", flush=True)
        print(f"Resuming from: {last_ckpt}", flush=True)

        model = YOLO(str(last_ckpt))
        model.train(resume=True, workers=0, device=DEVICE)

    else:
        print("\nSTARTING NEW TRAINING", flush=True)
        print(f"Starting from base weights: {MODEL_NAME}", flush=True)

        model = YOLO(MODEL_NAME)
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            project=str(MODELS_ROOT),
            name=RUN_NAME,
            val=False,
            workers=WORKERS,
            device=DEVICE,
            verbose=True,
            exist_ok=True,
        )

    print("Training call finished", flush=True)

    if best_ckpt.exists():
        return best_ckpt
    if last_ckpt.exists():
        return last_ckpt

    raise FileNotFoundError(f"No trained weights found in {weights_dir}")



In [43]:

# =========================================================
# MAIN
# =========================================================
def main():
    print("===== SCRIPT STARTED =====", flush=True)

    validate_settings()
    ensure_exists(RAW_DATA_ROOT, "raw_data")
    MODELS_ROOT.mkdir(parents=True, exist_ok=True)

    pairs = find_pairs(RAW_DATA_ROOT)
    print(f"Found {len(pairs)} videos", flush=True)

    if len(pairs) == 0:
        raise RuntimeError(f"No video/annotation pairs found in {RAW_DATA_ROOT}")

    if dataset_ready(DATASET_ROOT):
        print("Reusing dataset", flush=True)
    else:
        extract_dataset(pairs, DATASET_ROOT)

    data_yaml = write_yaml(DATASET_ROOT)
    print(f"Using data.yaml: {data_yaml}", flush=True)

    model_path = train(data_yaml)

    print(f"\nDone. Model saved at: {model_path}", flush=True)


if __name__ == "__main__":
    main()

===== SCRIPT STARTED =====
Found 28 videos
Dataset ready: 16133 images
Reusing dataset
Using data.yaml: C:\Users\benne\Computer_Vision\BoxingPunchDetection\datasets\boxing_yolo26_full\data.yaml

Starting training function...
Model selected: yolov8s.pt
Run directory: C:\Users\benne\Computer_Vision\BoxingPunchDetection\yolo26_models\resume_yolov8s
Checking checkpoint: C:\Users\benne\Computer_Vision\BoxingPunchDetection\yolo26_models\resume_yolov8s\weights\last.pt

STARTING NEW TRAINING
Starting from base weights: yolov8s.pt
New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.38  Python-3.13.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip

KeyboardInterrupt: 